# AI Mineral Discovery Platform: Model Training & Evaluation

This notebook documents the training, tuning, and evaluation of the machine learning model for predicting the **Mineral Potential Score** using NGCM Geochemical Data, Geological/Lithology Data, and Mineral Occurrence Data.

In [1]:
import os
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Load Preprocessed Datasets

In [2]:
data_dir = "../data"
df_ngcm = pd.read_csv(os.path.join(data_dir, "ngcm.csv"))
df_geology = pd.read_csv(os.path.join(data_dir, "geology.csv"))
df_min_occ = pd.read_csv(os.path.join(data_dir, "mineral_occurrence.csv"))

# Merge geochemistry and geology on coordinates and unit
df = pd.merge(df_ngcm, df_geology, on=['latitude', 'longitude', 'geological_unit'])
print(f"Merged Dataset Shape: {df.shape}")

Merged Dataset Shape: (10026, 75)


## 2. Compute Target Variable: Mineral Potential Score
The score represents the proximity to known mineralization occurrences/mines. It is modeled using an exponential decay function:

$$Score = e^{-\frac{Distance (km)}{10.0}}$$

In [3]:
lat_ngcm = df['latitude'].values
lon_ngcm = df['longitude'].values
lat_min = df_min_occ['y'].values
lon_min = df_min_occ['x'].values

min_dists_km = []
for lat, lon in zip(lat_ngcm, lon_ngcm):
    dists = np.sqrt((lat - lat_min)**2 + (lon - lon_min)**2) * 111.0
    min_dists_km.append(np.min(dists))
    
df['nearest_mineral_dist_km'] = min_dists_km
df['mineral_potential_score'] = np.exp(-df['nearest_mineral_dist_km'] / 10.0)

print(df['mineral_potential_score'].describe())

count    1.002600e+04
mean     6.369919e-02
std      1.419349e-01
min      5.234442e-08
25%      4.045132e-04
50%      6.113044e-03
75%      4.595903e-02
max      9.878514e-01
Name: mineral_potential_score, dtype: float64


## 3. Train-Test Split (80% / 20%)

In [4]:
geo_features = [c for c in df_ngcm.columns if c.endswith('_ppm') or c.endswith('_ppb') or c.endswith('__') or c.endswith('_') or c.endswith('loi')]
cat_features = ['rock_type', 'geological_unit', 'district']
features = geo_features + cat_features
target = 'mineral_potential_score'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Training size: {X_train.shape[0]}, Testing size: {X_test.shape[0]}")

Training size: 8020, Testing size: 2006


## 4. Pipeline Setup & Initial Model Evaluation

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), geo_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ])

models = {
    "Random Forest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    print(f"{name} R2 Score: {r2:.4f}")

Random Forest R2 Score: 0.8553
XGBoost R2 Score: 0.8523
Gradient Boosting R2 Score: 0.7719


## 5. Hyperparameter Tuning via GridSearchCV (Optimizing Random Forest)

In [ ]:
rf_reg = RandomForestRegressor(random_state=42, n_jobs=-1)
pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', rf_reg)])

param_grid = {
    'regressor__n_estimators': [100, 150],
    'regressor__max_depth': [15, None],
    'regressor__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='r2', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)
print(f"Best Parameters: {grid_search.best_params_}")

Fitting 3 folds for each of 8 candidates, totalling 24 fits


## 6. Final Model Evaluation & Serialization

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print(f"Final R2 Score: {r2_score(y_test, y_pred):.5f}")
print(f"Final RMSE:     {np.sqrt(mean_squared_error(y_test, y_pred)):.5f}")
print(f"Final MAE:      {mean_absolute_error(y_test, y_pred):.5f}")

# Serialize model
joblib.dump(best_model, "../models/best_model.pkl")